In [1]:
import pandas as pd
from datetime import datetime


In [2]:
# Revocation list obtained from: https://apps.irs.gov/pub/epostcard/data-download-revocation.zip 
# Load revocation list
revoked = pd.read_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/data-download-revocation.txt', sep='|', header=None, dtype=str)
revoked.columns = ['EIN', 'Legal_Name', 'DBA', 'Address', 'City', 'State', 
                   'ZIP', 'Country', 'Exemption_Type', 'Revocation_Date',
                   'Posting_Date', 'Reinstatement_Date']

In [3]:
revoked.head()

,EIN,Legal_Name,DBA,Address,City,State,ZIP,Country,Exemption_Type,Revocation_Date,Posting_Date,Reinstatement_Date
0,000003154,OAKLEAF FOREST TENANT MANAGEMENT,NaN,1706 GREENLEAF DR,NORFOLK,VA,23523-2112,US,03,15-NOV-2017,12-MAR-2018,NaN
1,000004101,SOUTH LAFOURCHE QUARTERBACK CLUB,NaN,167 BENT CYPRESS LN,LOCKPORT,LA,70374-4284,US,03,15-MAY-2023,14-AUG-2023,NaN
2,000065837,OHIO STATE GRANGE OF PATRONS OF HUSBANDRY,64 HURON COUNTY POMONA,4655 STATE ROUTE 60,WAKEMAN,OH,44889-8602,US,05,15-FEB-2015,11-MAY-2015,NaN
3,000107893,INTERNATIONAL ASSOCIATION OF LIONS,CLUBSSTURBRIDGE LION,204 OLD SPRINGFIELD RD,STAFFORD SPGS,CT,06076-3016,US,04,15-NOV-2010,13-JUL-2011,NaN
4,000262358,KEY CLUB INTERNATIONAL,H90622 HANFORD WEST HIGH SCHOOL,1150 W LACEY BLVD,HANFORD,CA,93230-3575,US,04,15-FEB-2011,09-NOV-2011,NaN


In [4]:
# Filter to Colorado and count
revoked_co = revoked[revoked['State'] == 'CO']
print(f"Colorado revocations: {len(revoked_co):,}")
revoked_co.to_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/revoked_co.csv', index = False)

# Count number of reinstatements - these are edge cases
reinstated_num = revoked_co['Reinstatement_Date'].count()
print(f"Colorado reinstatements: {reinstated_num}")

Colorado revocations: 21,851
Colorado reinstatements: 2955


In [5]:
# 990 data obtained from: https://990data.givingtuesday.org/datamarts/?co-item=basic-fields-679696bc141dbd17de689e0f
# 990EZ data obtained from: https://990data.givingtuesday.org/datamarts/?co-item=basic-fields-67cefae870c03882bcff2a64

# Load tax info
desired_990_cols = ['FILERUSSTATE', 'FILEREIN', 'TAXYEAR', 'ADDRESCHANGE', 'FILERUSCITY', 'FILERUSZIP', 'INITIARETURN', 'FINALRRETURN', 'AMENDERETURN', 'RETURNTYPE', 'URL', 'NAFBEOY', 'TOASEOOYY', 'TOLIEOOYY', 'TOTREVCURYEA', 'TOTEXPCURYEA', 'PROGSERVEXPE', 'CNIBEOY', 'STCIEOY', 'GRPAEOOYY', 'UNNEASEOOYY', 'TRNAEOY', 'PRNAEOY']
desired_990EZ_cols = ['FILERUSSTATE', 'FILEREIN', 'TAXYEAR', 'ADDRESCHANGE', 'FILERUSCITY', 'FILERUSZIP', 'INITIARETURN', 'FINALRRETURN', 'AMENDERETURN', 'RETURNTYPE', 'URL', 'NAFBEOY', 'TOASEOOYY', 'TOLIEOOYY', 'TOTALRREVENU', 'TOTALEEXPENS', 'CASAINEOOYY', 'TOTPROSEREXP']

tax990 = pd.read_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/2025_10_18_All_Years_990StandardFields.csv', usecols = desired_990_cols)
tax990EZ = pd.read_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/2025_10_28_All_Years_990EZStandardFields.csv', usecols = desired_990EZ_cols)

print(tax990.columns)
print(tax990EZ.columns)



Index(['RETURNTYPE', 'FILEREIN', 'TAXYEAR', 'ADDRESCHANGE', 'INITIARETURN',
       'FINALRRETURN', 'AMENDERETURN', 'FILERUSCITY', 'FILERUSSTATE',
       'FILERUSZIP', 'TOTREVCURYEA', 'TOTEXPCURYEA', 'TOASEOOYY', 'TOLIEOOYY',
       'NAFBEOY', 'PROGSERVEXPE', 'CNIBEOY', 'STCIEOY', 'GRPAEOOYY',
       'UNNEASEOOYY', 'TRNAEOY', 'PRNAEOY', 'URL'],
      dtype='object')
Index(['RETURNTYPE', 'TAXYEAR', 'ADDRESCHANGE', 'INITIARETURN', 'FINALRRETURN',
       'AMENDERETURN', 'FILERUSCITY', 'FILERUSSTATE', 'FILERUSZIP', 'FILEREIN',
       'TOTALRREVENU', 'TOTALEEXPENS', 'NAFBEOY', 'CASAINEOOYY', 'TOASEOOYY',
       'TOLIEOOYY', 'TOTPROSEREXP', 'URL'],
      dtype='object')


In [6]:
# Narrowing down the 990 and 990EZ to Colorado
tax990_co = tax990[tax990['FILERUSSTATE'] == 'CO']
tax990EZ_co = tax990EZ[tax990EZ['FILERUSSTATE'] == 'CO']
tax990_co_1M = tax990_co[tax990_co['TOTREVCURYEA'] < 1000000]

print(f"Colorado 990s: {len(tax990_co):,}")
print(f"Colorado 990EZs: {len(tax990EZ_co):,}")
print(f"Colorado 990s < $1MM: {len(tax990_co_1M):,}")

Colorado 990s: 77,259
Colorado 990EZs: 41,130
Colorado 990s < $1MM: 50,920


In [7]:
tax990_co_1M = tax990_co_1M.astype({'RETURNTYPE': 'str', 'FILEREIN': 'str'})
tax990EZ_co = tax990EZ_co.astype({'FILEREIN': 'str'})
tax990_co_1M['TAXYEAR'] = pd.to_datetime(tax990_co_1M['TAXYEAR'].astype(int).astype(str))
tax990EZ_co['TAXYEAR'] = pd.to_datetime(tax990EZ_co['TAXYEAR'].astype(int).astype(str))  


In [8]:
#tax990_co_1M.head()
#tax990_co_1M.describe()
#tax990EZ_co.head()
#tax990EZ_co.describe()
print(tax990_co_1M.info())
print(tax990EZ_co.info())


<class 'pandas.core.frame.DataFrame'>
Index: 50920 entries, 20 to 3643679
Data columns (total 23 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   RETURNTYPE    50920 non-null  object        
 1   FILEREIN      50920 non-null  object        
 2   TAXYEAR       50920 non-null  datetime64[ns]
 3   ADDRESCHANGE  2802 non-null   object        
 4   INITIARETURN  990 non-null    object        
 5   FINALRRETURN  470 non-null    object        
 6   AMENDERETURN  568 non-null    object        
 7   FILERUSCITY   50920 non-null  object        
 8   FILERUSSTATE  50920 non-null  object        
 9   FILERUSZIP    50920 non-null  float64       
 10  TOTREVCURYEA  50920 non-null  int64         
 11  TOTEXPCURYEA  50920 non-null  int64         
 12  TOASEOOYY     50920 non-null  int64         
 13  TOLIEOOYY     50920 non-null  int64         
 14  NAFBEOY       50920 non-null  int64         
 15  PROGSERVEXPE  48868 non-null  float64 

In [9]:
# Export EIN, Tax Year, URL
tax990_co_1M.to_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/tax990.csv', columns=['FILEREIN', 'TAXYEAR', 'URL'], index=False)
tax990EZ_co.to_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/tax990EZ.csv', columns = ['FILEREIN', 'TAXYEAR', 'URL'], index=False)


In [10]:

# load csvs
df_990 = pd.read_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/tax990.csv')
df_990ez = pd.read_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/tax990EZ.csv')

# Combine and deduplicate EINs
all_eins = pd.concat([df_990[['FILEREIN']], df_990ez[['FILEREIN']]]).drop_duplicates()

# Save to CSV
all_eins.to_csv('~/Documents/Nonprofit Project Plan/Nonprofit-Risk/Data/unique_eins.csv', index=False)
print(f"Saved {len(all_eins):,} unique EINs")

all_eins.head()

Saved 15,497 unique EINs


,FILEREIN
0,841231964
1,364483959
2,841559172
3,840707584
4,840762185
